# 03 · A controlled masked JEPA comparison

Does time-faithful preparation help while model capacity and exposure remain controlled?

These notebooks call the tested package; they contain no separate training loop. Real mode defaults to inspection. Explicit synthetic configuration plus TG_EXECUTE=1 runs software fixtures only.

[Notebook guide](README.md) · [HAIC runbook](../../docs/studies/temporal-gait/execution/haic.md)

In [ ]:
from pathlib import Path
import json
import os
import sys
from IPython.display import Markdown, display

project = os.environ.get("GAVD6_ROOT")
if project is None:
    project = next((str(p) for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "src/gavd6_sjepa").is_dir()), None)
if project is None:
    raise FileNotFoundError("Set GAVD6_ROOT to the gavd6 checkout.")
PROJECT_ROOT = Path(project).expanduser().resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from gavd6_sjepa.research_directions.temporal_gait.config import RunConfig
from gavd6_sjepa.research_directions.temporal_gait import workflow

cfg = RunConfig.from_env()
cfg.validate(check_input_paths=False)
phase = os.environ.get("TG_PHASE", "pilot")
execute = os.environ.get("TG_EXECUTE") == "1"
task_id = int(os.environ["TG_TASK_ID"]) if "TG_TASK_ID" in os.environ else None
print(f"Mode: {cfg.mode}; phase: {phase}; execute: {execute}; root: {cfg.root}")
if cfg.mode == "synthetic":
    display(Markdown("**SYNTHETIC SOFTWARE FIXTURE — no real GAVD result.**"))
if not execute:
    display(Markdown("**PLAN ONLY.** Set TG_EXECUTE=1 only to run this selected stage."))

## 1. The controlled question

Compare masked_index and masked on identical raw windows. The first mechanism comparison uses explicitly audited historical-overlap sources and keeps width 96, four encoder layers, two predictor layers, four-sample patches and the same recipe. Full-allowed cohort expansion, clock channels and two-sample patches are subsequent named comparisons in separate run roots.

In [ ]:
print(json.dumps(cfg.to_dict(), indent=2, default=str))
tasks = workflow.plan_tasks(cfg, phase)
print(json.dumps(tasks, indent=2, default=str))

## 2. Execute only the selected stage

The configured task ID refers to one immutable arm/seed row. A normal notebook render does not train. Slurm and the command-line runner pass the same stage and phase.

In [ ]:
stage = os.environ.get("TG_STAGE", 'masked')
if stage not in ('masked',):
    raise ValueError(f"This notebook cannot execute stage {stage}")
if execute:
    if stage in ("masked", "future") and task_id is None:
        raise ValueError("Choose one TG_TASK_ID from the immutable phase task plan.")
    result = workflow.run_stage(cfg, stage, task_id=task_id,
                                role="test" if stage == "test" else "development",
                                phase=phase)
    if result.get("status") in {"incomplete", "failed", "error"}:
        print(json.dumps(result, indent=2, default=str))
        raise RuntimeError(f"Stage did not complete: {result['status']}; inspect retained artifacts")
else:
    result = {"status": "plan_only", "stage": stage, "mode": cfg.mode,
              "phase": phase, "task_id": task_id, "media_opened": False}

## 3. Inspect retained evidence

Inspect the selected global task ID, seed, realized masks, context/target support, initialization/online/teacher checkpoints and learning curves.

In [ ]:
print(json.dumps(result, indent=2, default=str))
print('This record describes execution/artifacts; scientific conclusions require the saved comparison.')

## 4. Interpretation and next decision

Prepared-feature masking is not raw-observation withholding. This bidirectional-prefix objective is not future-only pretraining; lower teacher loss is not the success metric.